In [43]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df=pd.read_csv('../../Datasets/Titanic-Dataset.csv')
print(df.sample(5))

     PassengerId  Survived  Pclass                                 Name  \
647          648         1       1  Simonius-Blumer, Col. Oberst Alfons   
303          304         1       2                  Keane, Miss. Nora A   
711          712         0       1                   Klaber, Mr. Herman   
463          464         0       2         Milling, Mr. Jacob Christian   
250          251         0       3               Reed, Mr. James George   

        Sex   Age  SibSp  Parch  Ticket   Fare Cabin Embarked  
647    male  56.0      0      0   13213  35.50   A26        C  
303  female   NaN      0      0  226593  12.35  E101        Q  
711    male   NaN      0      0  113028  26.55  C124        S  
463    male  48.0      0      0  234360  13.00   NaN        S  
250    male   NaN      0      0  362316   7.25   NaN        S  


> - #### Drop Unwanted Columns : PassengerId, Name, Ticket, Cabin,Age,Pclass,Fare

In [44]:
# Drop PassengerId, Name, Ticket, Cabin,'Age','Pclass','SibSp','Parch','Fare'
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin','Age','Pclass','SibSp','Parch','Fare'], axis=1, inplace=True)


>  - ####  Impute the missing and unknown values using the SimpleImputer instance 
>  - ####  Perform Test Train Split

In [45]:

df.sample(5)

imputer=SimpleImputer(strategy='most_frequent') 

df['Embarked']=imputer.fit_transform(df['Embarked'].values.reshape(-1,1)) #Shape should be reshaped while giving to the imputer because it only accepts 2D arrays
print(df.sample(5))
x=df[['Sex','Embarked']]
y=df['Survived']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


     Survived   Sex Embarked
222         0  male        S
411         0  male        Q
260         0  male        Q
739         0  male        S
23          1  male        S


### One Hot Encoding
    - Initially we are not dropping one column 

In [46]:
ohe = OneHotEncoder(sparse=False) #sparse=False will return a numpy array
x_train_encoded=ohe.fit_transform(x_train)
x_train_encoded_df = pd.DataFrame(
    x_train_encoded,
    index=x_train.index,
    columns=ohe.get_feature_names_out(x_train.columns)
)
print(x_train_encoded_df.sample(5))
x_test_encoded = ohe.transform(x_test)
x_test_encoded_df = pd.DataFrame(
    x_test_encoded,
    index=x_test.index,
    columns=ohe.get_feature_names_out(x_train.columns)
)
print(x_test_encoded_df.sample(5))

     Sex_female  Sex_male  Embarked_C  Embarked_Q  Embarked_S
612         1.0       0.0         0.0         1.0         0.0
262         0.0       1.0         0.0         0.0         1.0
133         1.0       0.0         0.0         0.0         1.0
654         1.0       0.0         0.0         1.0         0.0
803         0.0       1.0         1.0         0.0         0.0
     Sex_female  Sex_male  Embarked_C  Embarked_Q  Embarked_S
538         0.0       1.0         0.0         0.0         1.0
446         1.0       0.0         0.0         0.0         1.0
462         0.0       1.0         0.0         0.0         1.0
70          0.0       1.0         0.0         0.0         1.0
299         1.0       0.0         1.0         0.0         0.0


c:\Users\dell\miniconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:828: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


### K-1 Hot Encoding To prevent Multicollinearity

In [48]:
ohe2 = OneHotEncoder(drop='first',sparse=False) #sparse=False will return a numpy array
x_train_Encoded=ohe2.fit_transform(x_train)
x_train_Encoded_df=pd.DataFrame(
    x_train_Encoded,
    index=x_train.index,
    columns=ohe2.get_feature_names_out(x_train.columns)
)
print(x_train_Encoded_df.sample(5))

     Sex_male  Embarked_Q  Embarked_S
575       1.0         0.0         1.0
361       1.0         0.0         0.0
752       1.0         0.0         1.0
730       0.0         0.0         1.0
663       1.0         0.0         1.0


c:\Users\dell\miniconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:828: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(



### Comparison of OneHotEncoder with and without drop parameter

#### Without drop parameter (Original OneHotEncoder)
- Creates binary columns for all unique categories
- For 'Sex' feature: Creates 2 columns (female, male)
- For 'Embarked' feature: Creates 3 columns (C, Q, S)
- Total features created: 5 columns
- May lead to multicollinearity due to perfect correlation between features
- Acceptable in case of the nonlinear models but not in the case linear models

#### With drop='first' parameter (K-1 OneHotEncoder) 
- Drops first category for each feature to avoid multicollinearity
- For 'Sex' feature: Only keeps 'male' column (female=0, male=1)
- For 'Embarked' feature: Drops 'C', keeps 'Q' and 'S' columns
- Total features created: 3 columns
- Prevents multicollinearity while preserving all information
